# Checkpoint 3 — UKDALE_01 Kernel SHAP seed-42 pilot

**One controlled reconstruction only.** The code checks the historical environment, model, archived input, training-row order, background hashes and expectation before it runs exactly one Kernel SHAP explanation. It does not retrain models or overwrite any files. If any prerequisite fails, it stops.

In [ ]:
# CHECKPOINT 3 — one-window Kernel SHAP reproduction (no writes/retraining)
from pathlib import Path
import hashlib, importlib.metadata as im, json
import numpy as np
import pandas as pd

# Preserve the original runtime evidence; do not silently substitute another stack.
REQUIRED = {'scikit-learn': '1.6.1', 'shap': '0.54.1', 'numpy': '2.4.6', 'pandas': '2.2.3'}
for library, expected in REQUIRED.items():
    actual = im.version(library)
    print(f'ENV {library}: {actual} (expected {expected})')
    if actual != expected:
        raise RuntimeError(f'STOP_ENV_VERSION: {library} does not match verified D4 runtime')

from google.colab import drive
import joblib, shap
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')

ROOT = Path('/content/drive/MyDrive/Master/Thesis/Code')
D5 = ROOT / 'experiments/D5_C3_12_C4_12_LOCAL_XAI_EVIDENCE_20261106T183849_627223Z'
MODEL = ROOT / 'SUPERVISOR_FINAL_EXPERIMENTS_2026/03_integrity_checks/IF_RECOVERY/UKDALE/ukdale_full_train_IF_FUNCTIONALLY_VERIFIED_20260928T180619_923276Z.joblib'
SCORES = ROOT / 'experiments/EXP_SPLIT_BOUNDARY_PURGE_CORRECTION/UKDALE/ukdale_if_scored_boundary_corrected.csv'
INPUTS = D5 / 'selected_scaled_inputs_13_windows.csv'
ARCHIVE = D5 / 'historical_snapshots/ukdale_historical_shap.csv'

FEATURES = ['energy','first_last_diff','iqr','kurtosis','mad','max',
            'max_abs_diff','mean','mean_abs_diff','median','min','n_points',
            'peak_count','q25','q75','range','rms','skew','slope','std',
            'std_diff','sum','var']
COLUMNS = [f'{f}_z' for f in FEATURES]
EXPECTED = {
    'model_hash': 'c1695e8c4265f6e90994be88ea7ec645acba8e10ed6240697aa73703ce3c9c5b',
    'input_hash': '89fc3c115cd2e9873b1828c06d7a612ed9da1e0b760e7948f40f279f63e3caa8',
    'archive_hash': 'cddc6839dacd728bcf3be917b5ad8f86e7c3411c4bc0278610d152d1cbe801be',
    'background_hash': '5ec3aadf7041920574ddbbe4992a30c24bad414d2c9426d4ab56309ff7acfc74',
    'positions_hash': 'd6f7c541a106a6f585a999073e152e70c2dc9fc54d72b48afef181b032b9e641',
    'expected_value': -0.1271061828008163,
}

def require(condition, label):
    if not bool(condition):
        raise RuntimeError('FAIL / STOP: ' + label)
    print('PASS:', label)

def file_hash(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

for p in (MODEL, SCORES, INPUTS, ARCHIVE):
    require(p.is_file(), f'file present: {p.name}')
for p, expected in [(MODEL, EXPECTED['model_hash']),
                    (INPUTS, EXPECTED['input_hash']),
                    (ARCHIVE, EXPECTED['archive_hash'])]:
    require(file_hash(p) == expected, f'SHA-256: {p.name}')

selected = pd.read_csv(INPUTS, float_precision='round_trip')
case = selected.loc[selected['window_id'].eq('UKDALE_01')]
require(len(selected) == 13 and len(case) == 1, '13-window evidence / unique pilot')
x = case[COLUMNS].to_numpy(dtype=np.float32)
require(x.shape == (1, 23) and np.isfinite(x).all(), 'pilot input shape/dtype/finite')

model = joblib.load(MODEL)  # Previously verified user-owned artifact.
require(model.n_features_in_ == 23 and len(model.estimators_) == 300 and model.max_samples_ == 256,
        'frozen model structure')
score_fn = lambda X: -model.decision_function(np.asarray(X, dtype=np.float32))
score = float(score_fn(x)[0])
require(abs(score - float(case.iloc[0]['anomaly_score'])) <= 1e-9,
        'model reproduces archived UKDALE_01 anomaly score')
print('Model score:', f'{score:.17f}')

# Read only the required CSV columns. UK training order MUST be source-row ID.
usecols = ['split', 'original_source_row_id', 'anomaly_score'] + COLUMNS
pieces = []
for chunk in pd.read_csv(SCORES, usecols=usecols, chunksize=25000,
                         float_precision='round_trip', low_memory=False):
    part = chunk.loc[chunk['split'].eq('train')].copy()
    if len(part):
        pieces.append(part)
train = pd.concat(pieces, ignore_index=True)
require(len(train) == 218358, 'frozen UK-DALE training population')
ids = pd.to_numeric(train['original_source_row_id'], errors='coerce')
require(ids.notna().all() and ids.is_unique, 'unique, numeric original_source_row_id')
train = train.assign(_source_id_numeric=ids).sort_values('_source_id_numeric', kind='stable')
X_train = train[COLUMNS].to_numpy(dtype=np.float32)
require(X_train.shape == (218358, 23) and np.isfinite(X_train).all(),
        'ordered 23-feature training representation')

positions = np.random.default_rng(42).choice(len(X_train), size=100, replace=False)
background = X_train[positions].copy()
hash_bg = hashlib.sha256(background.tobytes(order='C')).hexdigest()
hash_pos = hashlib.sha256(np.asarray(positions, dtype='<i8').tobytes()).hexdigest()
require(hash_bg == EXPECTED['background_hash'], 'historical 100-row SHAP background hash')
require(hash_pos == EXPECTED['positions_hash'], 'historical background index hash')
ref_scores = train['anomaly_score'].to_numpy(dtype=np.float64)[positions]
bg_scores = np.asarray(score_fn(background), dtype=np.float64)
require(np.max(np.abs(bg_scores - ref_scores)) <= 1e-9, '100 archived background scores')
require(abs(float(bg_scores.mean()) - EXPECTED['expected_value']) <= 1e-9,
        'SHAP background expected value')
print('Background mean:', f'{bg_scores.mean():.17f}')

hist = pd.read_csv(ARCHIVE, float_precision='round_trip')
mask = (hist['building'].eq(1) & hist['meter'].eq(6) &
        pd.to_datetime(hist['window_start'], utc=True).eq(pd.Timestamp('2016-08-15 21:45:00+00:00')))
hist = hist.loc[mask].copy()
require(len(hist) == 23 and set(hist['feature']) == set(FEATURES),
        'historical UKDALE_01 SHAP complete 23-feature vector')
hist = hist.set_index('feature').loc[FEATURES]
old_values = hist['shap_value'].to_numpy(dtype=np.float64)
old_x = hist['feature_value_scaled'].to_numpy(dtype=np.float64)
require(np.max(np.abs(old_x - x[0])) <= 1e-5, 'historical input agrees within recorded precision')
require(abs(EXPECTED['expected_value'] + old_values.sum() - score) <= 1e-9,
        'archived SHAP additivity')

print('\n--- All prerequisites passed. Running ONE Kernel SHAP explanation. ---')
# The original historical coalition RNG state was NOT recorded.
# This fixed seed defines a new controlled reconstruction, not guaranteed historical identity.
np.random.seed(42)
explainer = shap.KernelExplainer(score_fn, background)
new_values = np.asarray(explainer.shap_values(x, nsamples=300), dtype=np.float64).reshape(-1)
require(new_values.shape == (23,) and np.isfinite(new_values).all(),
        'fresh 23-feature Kernel SHAP output')
new_base = float(np.asarray(explainer.expected_value).reshape(-1)[0])
require(abs(new_base - EXPECTED['expected_value']) <= 1e-9, 'fresh SHAP expected value')
residual = abs(new_base + new_values.sum() - score)
require(residual <= 1e-6, 'fresh SHAP additivity (<=1e-6)')

abs_difference = np.abs(new_values - old_values)
old_top5 = [FEATURES[i] for i in np.argsort(-np.abs(old_values), kind='stable')[:5]]
new_top5 = [FEATURES[i] for i in np.argsort(-np.abs(new_values), kind='stable')[:5]]
print('\n=== UKDALE_01 REPRODUCTION REPORT ===')
print('score:', score)
print('fresh baseline:', new_base)
print('additivity residual:', residual)
print('maximum absolute 23-vector difference:', float(abs_difference.max()))
print('mean absolute 23-vector difference:', float(abs_difference.mean()))
print('historical top five:', old_top5)
print('fresh top five:', new_top5)
print('top-five Jaccard:', len(set(old_top5) & set(new_top5))/len(set(old_top5)|set(new_top5)))
print(pd.DataFrame({'feature':FEATURES, 'archived_shap':old_values,
                    'fresh_shap':new_values, 'absolute_difference':abs_difference})
      .to_string(index=False))
if abs_difference.max() <= 1e-6:
    print('PILOT STATUS: FULL_VECTOR_NUMERIC_MATCH (<=1e-6)')
else:
    print('PILOT STATUS: CONTROLLED_RECONSTRUCTION_NOT_EXACT_HISTORICAL_MATCH')
    print('STOP before five-seed work; investigate original coalition RNG / SHAP version.')
print('No models fitted; no historical artifacts written.')
